# Student-t HMM regime consensus

Multiple assets contribute regime signals from Student-t hidden Markov models. A walk-forward backtest evaluates the consensus rule and plots its performance.

Daily OHLCV is downloaded from Yahoo Finance when the notebook runs. Open this notebook from the `notebooks/` directory so it can import the shared data loader.


In [ ]:
"""
MULTI-ASSET REGIME SYNCHRONIZATION - CONCEPTUALLY CORRECTED
===========================================================

CRITICAL CONCEPTUAL FIXES:
-------------------------
1. Regime labels use QQQ forward returns (not each asset's own returns)
2. HMM trains on multi-feature vector [return, vol, momentum]
3. Consensus weighted by historical correlation to QQQ
4. Signal horizon = rebalance frequency (aligned at 1 day)
5. Benchmark = QQQ (matches trading instrument)
6. Entry price tracking for accurate stop-loss

LOOK-AHEAD & BIAS FIXES (V4):
------------------------------
7. Stop-loss: Evaluate using Close[T] (known) not Open[T+1] (unknown at signal time)
8. Consensus weights: Use holdout portion (last 20% of training) for IR calculation
   instead of in-sample IR which overfits to training data
9. Feature double-shift removed: YZ vol and momentum already computed from lagged OHLC,
   the extra .shift(1) caused unnecessary 2-day lag (T-2) when T-1 suffices
"""

from _market_data import download_ohlcv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime, timedelta
from typing import Dict, Tuple, List, Optional
from dataclasses import dataclass
from sklearn.preprocessing import StandardScaler
from scipy.stats import t as student_t
from scipy.optimize import minimize
from scipy.special import gammaln

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

class StrategyConfig:
    
    BACKTEST_START_DATE = "2013-01-01"
    BACKTEST_END_DATE = "2018-12-31"
    
    INITIAL_CAPITAL = 100000.0
    BENCHMARK_TICKER = "QQQ"  # FIXED: Match trading instrument
    
    MACRO_ASSETS = ["SPY", "TLT", "HYG"]
    LONG_TICKER = "QQQ"
    HEDGE_TICKER = "SH"
    
    # HMM Parameters
    HMM_N_STATES = 2
    HMM_N_ITER = 150
    HMM_RANDOM_STATE = 42
    HMM_CONVERGENCE_TOL = 1e-4
    
    # Student-t parameters
    STUDENT_T_DF_INIT = 5.0
    STUDENT_T_DF_MIN = 2.1
    STUDENT_T_DF_MAX = 30.0
    
    # Feature parameters
    VOL_WINDOW = 20
    MOMENTUM_WINDOW = 21
    
    # Walk-forward parameters
    MIN_TRAINING_DAYS = 252
    REFIT_FREQUENCY = 63
    REBALANCE_FREQUENCY = 1  # FIXED: Daily to match 1-day signal horizon
    
    # FIXED: Consensus thresholds (used in both logic and visualization)
    CONSENSUS_RISK_ON_THRESHOLD = 0.55
    CONSENSUS_RISK_OFF_THRESHOLD = 0.45
    
    # Graduated exposure levels
    STRONG_RISK_ON_EXPOSURE = 1.20
    MODERATE_RISK_ON_EXPOSURE = 0.90
    NEUTRAL_EXPOSURE = 0.60
    DEFENSIVE_EXPOSURE = 0.30
    STRONG_RISK_OFF_HEDGE = 0.10
    
    # Transaction costs
    COMMISSION_RATE = 0.0005
    BID_ASK_SPREAD = 0.0003
    MARKET_IMPACT = 0.0002
    MIN_COMMISSION = 1.0
    
    # Stop loss
    STOP_LOSS_ENABLED = True
    STOP_LOSS_METHOD = "cvar95"
    CVAR_CONFIDENCE = 0.95
    CVAR_LOOKBACK = 60
    STOP_LOSS_BUFFER = 0.80
    
    # Risk parameters
    RISK_FREE_RATE = 0.02
    MARGIN_RATE_ANNUAL = 0.03
    MAX_LEVERAGE = 1.50


# ============================================================================
# STUDENT-T HMM (Multi-feature capable)
# ============================================================================

class StudentTHMM:
    """Student-t HMM supporting multi-dimensional features."""
    
    def __init__(self, n_states: int = 2, n_iter: int = 150,
                 tol: float = 1e-4, random_state: int = 42,
                 df_init: float = 5.0, df_min: float = 2.1, df_max: float = 30.0):
        self.n_states = n_states
        self.n_iter = n_iter
        self.tol = tol
        self.random_state = random_state
        self.df_init = df_init
        self.df_min = df_min
        self.df_max = df_max
        
        self.transmat_ = None
        self.startprob_ = None
        self.means_ = None  # Now (n_states, n_features)
        self.covars_ = None  # Now (n_states, n_features, n_features)
        self.dfs_ = None
        
        self.converged_ = False
        self.n_iter_ = 0
        self.n_features_ = None
    
    def _multivariate_student_t_logpdf(self, X: np.ndarray, mu: np.ndarray, 
                                       Sigma: np.ndarray, df: float) -> np.ndarray:
        """Multivariate Student-t log PDF."""
        n, d = X.shape
        
        # Mahalanobis distance
        diff = X - mu
        try:
            L = np.linalg.cholesky(Sigma)
            z = np.linalg.solve(L, diff.T).T
            maha_dist = np.sum(z**2, axis=1)
        except:
            # Fallback to pseudo-inverse if singular
            Sigma_inv = np.linalg.pinv(Sigma)
            maha_dist = np.sum(diff @ Sigma_inv * diff, axis=1)
        
        # Log PDF
        log_norm = (gammaln((df + d) / 2) - gammaln(df / 2) 
                   - (d / 2) * np.log(np.pi * df)
                   - 0.5 * np.linalg.slogdet(Sigma)[1])
        
        log_prob = log_norm - ((df + d) / 2) * np.log(1 + maha_dist / df)
        
        return log_prob
    
    def _initialize_parameters(self, X: np.ndarray):
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        self.n_features_ = n_features
        
        self.startprob_ = np.ones(self.n_states) / self.n_states
        self.transmat_ = np.ones((self.n_states, self.n_states)) / self.n_states
        for i in range(self.n_states):
            self.transmat_[i, i] += 0.1
        self.transmat_ = self.transmat_ / self.transmat_.sum(axis=1, keepdims=True)
        
        # K-means-like initialization
        quantiles = np.linspace(0, 1, self.n_states + 1)
        first_feature = X[:, 0]
        
        self.means_ = np.zeros((self.n_states, n_features))
        self.covars_ = np.zeros((self.n_states, n_features, n_features))
        self.dfs_ = np.full(self.n_states, self.df_init)
        
        for i in range(self.n_states):
            lower = np.quantile(first_feature, quantiles[i])
            upper = np.quantile(first_feature, quantiles[i + 1])
            mask = (first_feature >= lower) & (first_feature <= upper)
            
            if mask.sum() > 0:
                cluster_data = X[mask]
                self.means_[i] = cluster_data.mean(axis=0)
                cov = np.cov(cluster_data.T)
                # Regularize
                self.covars_[i] = cov + np.eye(n_features) * 1e-3
            else:
                self.means_[i] = X.mean(axis=0)
                self.covars_[i] = np.cov(X.T) + np.eye(n_features) * 1e-3
    
    def _forward(self, X: np.ndarray) -> Tuple[np.ndarray, float]:
        n_samples = len(X)
        
        log_B = np.zeros((n_samples, self.n_states))
        for i in range(self.n_states):
            log_B[:, i] = self._multivariate_student_t_logpdf(
                X, self.means_[i], self.covars_[i], self.dfs_[i]
            )
        
        log_alpha = np.zeros((n_samples, self.n_states))
        log_alpha[0] = np.log(self.startprob_) + log_B[0]
        
        for t in range(1, n_samples):
            for j in range(self.n_states):
                log_trans = np.log(self.transmat_[:, j])
                log_alpha[t, j] = self._log_sum_exp(
                    log_alpha[t-1] + log_trans
                ) + log_B[t, j]
        
        log_prob = self._log_sum_exp(log_alpha[-1])
        
        alpha = np.exp(log_alpha - log_alpha.max(axis=1, keepdims=True))
        alpha = alpha / alpha.sum(axis=1, keepdims=True)
        
        return alpha, log_prob
    
    def _backward(self, X: np.ndarray) -> np.ndarray:
        n_samples = len(X)
        
        log_B = np.zeros((n_samples, self.n_states))
        for i in range(self.n_states):
            log_B[:, i] = self._multivariate_student_t_logpdf(
                X, self.means_[i], self.covars_[i], self.dfs_[i]
            )
        
        log_beta = np.zeros((n_samples, self.n_states))
        # FIXED: Explicitly initialize β[T-1] = 1 in log-space (log(1) = 0)
        log_beta[-1, :] = 0.0
        
        for t in range(n_samples - 2, -1, -1):
            for i in range(self.n_states):
                log_trans = np.log(self.transmat_[i, :])
                log_beta[t, i] = self._log_sum_exp(
                    log_trans + log_B[t+1] + log_beta[t+1]
                )
        
        beta = np.exp(log_beta - log_beta.max(axis=1, keepdims=True))
        beta = beta / beta.sum(axis=1, keepdims=True)
        
        return beta
    
    def _log_sum_exp(self, log_values: np.ndarray) -> float:
        max_val = np.max(log_values)
        return max_val + np.log(np.sum(np.exp(log_values - max_val)))
    
    def _compute_posteriors(self, alpha: np.ndarray, 
                           beta: np.ndarray, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        n_samples = alpha.shape[0]
        
        gamma = alpha * beta
        gamma = gamma / gamma.sum(axis=1, keepdims=True)
        
        xi = np.zeros((n_samples - 1, self.n_states, self.n_states))
        
        for t in range(n_samples - 1):
            for i in range(self.n_states):
                for j in range(self.n_states):
                    log_B_next = self._multivariate_student_t_logpdf(
                        X[t+1:t+2], self.means_[j], self.covars_[j], self.dfs_[j]
                    )[0]
                    
                    xi[t, i, j] = (alpha[t, i] * 
                                  self.transmat_[i, j] * 
                                  np.exp(log_B_next) * 
                                  beta[t+1, j])
            
            xi_sum = xi[t].sum()
            if xi_sum > 0:
                xi[t] = xi[t] / xi_sum
        
        return gamma, xi
    
    def _m_step(self, X: np.ndarray, gamma: np.ndarray, xi: np.ndarray):
        from sklearn.covariance import LedoitWolf
        
        self.startprob_ = gamma[0] / gamma[0].sum()
        
        for i in range(self.n_states):
            denom = gamma[:-1, i].sum()
            if denom > 0:
                for j in range(self.n_states):
                    self.transmat_[i, j] = xi[:, i, j].sum() / denom
        
        for i in range(self.n_states):
            gamma_i = gamma[:, i]
            gamma_sum = gamma_i.sum()
            
            if gamma_sum > 1e-6:
                # Update mean
                self.means_[i] = (gamma_i[:, None] * X).sum(axis=0) / gamma_sum
                
                # Update covariance with Ledoit-Wolf shrinkage
                diff = X - self.means_[i]
                
                # FIXED: Apply Ledoit-Wolf shrinkage for stability
                weighted_cov = (diff.T @ (gamma_i[:, None] * diff)) / gamma_sum
                
                # Shrinkage: αF + (1-α)S where F is diagonal target
                try:
                    lw = LedoitWolf()
                    # Use unweighted data for shrinkage estimation
                    state_mask = gamma_i > 0.5 / len(gamma_i)
                    if state_mask.sum() > self.n_features_:
                        lw.fit(X[state_mask])
                        shrinkage = lw.shrinkage_
                        target = np.diag(np.diag(weighted_cov))
                        self.covars_[i] = shrinkage * target + (1 - shrinkage) * weighted_cov
                    else:
                        self.covars_[i] = weighted_cov + np.eye(self.n_features_) * 1e-3
                except:
                    # Fallback to simple regularization
                    self.covars_[i] = weighted_cov + np.eye(self.n_features_) * 1e-3
    
    def fit(self, X: np.ndarray) -> 'StudentTHMM':
        self._initialize_parameters(X)
        
        prev_log_prob = -np.inf
        
        for iteration in range(self.n_iter):
            alpha, log_prob = self._forward(X)
            beta = self._backward(X)
            gamma, xi = self._compute_posteriors(alpha, beta, X)
            
            self._m_step(X, gamma, xi)
            
            improvement = log_prob - prev_log_prob
            
            if iteration > 0 and improvement < self.tol:
                self.converged_ = True
                self.n_iter_ = iteration + 1
                break
            
            prev_log_prob = log_prob
        
        if not self.converged_:
            self.n_iter_ = self.n_iter
        
        return self
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        n_samples = len(X)
        
        log_B = np.zeros((n_samples, self.n_states))
        for i in range(self.n_states):
            log_B[:, i] = self._multivariate_student_t_logpdf(
                X, self.means_[i], self.covars_[i], self.dfs_[i]
            )
        
        log_delta = np.zeros((n_samples, self.n_states))
        psi = np.zeros((n_samples, self.n_states), dtype=int)
        
        log_delta[0] = np.log(self.startprob_) + log_B[0]
        
        for t in range(1, n_samples):
            for j in range(self.n_states):
                transition_probs = log_delta[t-1] + np.log(self.transmat_[:, j])
                psi[t, j] = np.argmax(transition_probs)
                log_delta[t, j] = transition_probs[psi[t, j]] + log_B[t, j]
        
        states = np.zeros(n_samples, dtype=int)
        states[-1] = np.argmax(log_delta[-1])
        
        for t in range(n_samples - 2, -1, -1):
            states[t] = psi[t + 1, states[t + 1]]
        
        return states
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        alpha, _ = self._forward(X)
        beta = self._backward(X)
        gamma, _ = self._compute_posteriors(alpha, beta, X)
        return gamma


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================

class FeatureEngine:
    """Feature engineering with proper Rogers-Satchell and double-lagging."""
    
    @staticmethod
    def calculate_yang_zhang_volatility(df: pd.DataFrame, window: int) -> pd.Series:
        O = df['Open']
        H = df['High']
        L = df['Low']
        C = df['Close']
        
        log_ho = np.log(H / O)
        log_hc = np.log(H / C)
        log_lo = np.log(L / O)
        log_lc = np.log(L / C)
        log_co = np.log(C / O)
        log_oc = np.log(O / C.shift(1))
        
        # Correct Rogers-Satchell
        rs_vol = log_ho * log_hc + log_lo * log_lc
        cc_vol = log_co ** 2
        oc_vol = log_oc ** 2
        
        k = 0.34 / (1.34 + (window + 1) / (window - 1))
        
        rs_roll = rs_vol.rolling(window, min_periods=window).mean()
        cc_roll = cc_vol.rolling(window, min_periods=window).mean()
        oc_roll = oc_vol.rolling(window, min_periods=window).mean()
        
        yz_vol = np.sqrt(oc_roll + k * cc_roll + (1 - k) * rs_roll) * np.sqrt(252)
        
        return yz_vol
    
    @staticmethod
    def calculate_momentum(returns: pd.Series, window: int) -> pd.Series:
        total_return = returns.rolling(window, min_periods=window).sum()
        volatility = returns.rolling(window, min_periods=window).std()
        momentum = total_return / (volatility + 1e-6)
        return momentum
    
    @staticmethod
    def prepare_features(df: pd.DataFrame, asset: str, config: StrategyConfig) -> pd.DataFrame:
        """
        FIXED: Prepare multi-feature vector with proper double-lagging.
        Returns: [return, volatility, momentum] all double-lagged.
        """
        features = df.copy()
        features = features.sort_values('Date').reset_index(drop=True)
        
        # Log returns
        features['log_return'] = np.log(features['Close'] / features['Close'].shift(1))
        features['log_return_lag1'] = features['log_return'].shift(1)
        
        # DOUBLE-SHIFT FIX: YZ vol computed on lagged OHLC already prevents look-ahead.
        # The extra .shift(1) was causing unnecessary 2-day lag (T-2 data instead of T-1).
        # Signal at Close[T] → execute at Open[T+1], so T-1 features are valid.
        
        # Volatility from lagged OHLC (uses data up to T-1 = known at Close[T])
        features['Open_lag1'] = features['Open'].shift(1)
        features['High_lag1'] = features['High'].shift(1)
        features['Low_lag1'] = features['Low'].shift(1)
        features['Close_lag1'] = features['Close'].shift(1)
        
        lagged_df = pd.DataFrame({
            'Open': features['Open_lag1'],
            'High': features['High_lag1'],
            'Low': features['Low_lag1'],
            'Close': features['Close_lag1']
        })
        
        # FIX: No extra shift - vol from lagged OHLC is already T-1 data
        features['volatility'] = FeatureEngine.calculate_yang_zhang_volatility(
            lagged_df, config.VOL_WINDOW
        )
        
        # FIX: No extra shift - momentum from lagged returns is already T-1 data
        features['momentum'] = FeatureEngine.calculate_momentum(
            features['log_return_lag1'], config.MOMENTUM_WINDOW
        )
        
        # Single-lagged features (T-1 data, known at Close[T])
        features[f'{asset}_return_lag1'] = features['log_return_lag1']
        features[f'{asset}_vol_lag1'] = features['volatility']
        features[f'{asset}_momentum_lag1'] = features['momentum']
        
        return features


# ============================================================================
# MULTI-ASSET DETECTOR (Conceptually Fixed)
# ============================================================================

@dataclass
class RegimeMapping:
    state_to_regime: Dict[int, str]
    mean_returns: Dict[int, float]
    training_end_date: pd.Timestamp
    asset: str
    correlation_with_target: float  # NEW: Track predictive power

class WalkForwardMultiAssetDetector:
    """
    FIXED CONCEPTS:
    1. Labels based on QQQ forward returns (not each asset's own)
    2. Multi-feature HMM [return, vol, momentum]
    3. Weighted consensus by correlation to QQQ
    """
    
    def __init__(self, config: StrategyConfig):
        self.config = config
        self.hmms = {}
        self.regime_mappings = {}
        self.fold_history = []
    
    def walk_forward_predict(self, features_dict: Dict[str, pd.DataFrame],
                            target_data: pd.DataFrame) -> pd.DataFrame:
        """
        FIXED: target_data is QQQ prices for forward return calculation.
        """
        print("\n" + "="*80)
        print("WALK-FORWARD MULTI-ASSET DETECTION (CONCEPTUALLY FIXED)")
        print("="*80)
        
        # Align dates
        common_dates = None
        for asset, df in features_dict.items():
            df = df.dropna(subset=[f'{asset}_return_lag1', 
                                  f'{asset}_vol_lag1', 
                                  f'{asset}_momentum_lag1']).copy()
            asset_dates = df['Date']
            if common_dates is None:
                common_dates = asset_dates
            else:
                common_dates = common_dates[common_dates.isin(asset_dates)]
        
        # Also align with target data
        target_dates = target_data['Date']
        common_dates = common_dates[common_dates.isin(target_dates)]
        common_dates = common_dates.sort_values().reset_index(drop=True)
        
        print(f"\nCommon dates: {len(common_dates)}")
        print(f"Range: {common_dates.min().date()} to {common_dates.max().date()}")
        
        # Initialize results
        results = pd.DataFrame({'Date': common_dates})
        for asset in features_dict.keys():
            results[f'{asset}_regime'] = np.nan
            results[f'{asset}_regime_name'] = None
            results[f'{asset}_prob_risk_on'] = np.nan
            results[f'{asset}_weight'] = np.nan  # NEW: Consensus weight
        
        results['consensus_probability'] = np.nan
        results['final_regime'] = None
        
        min_train = self.config.MIN_TRAINING_DAYS
        refit_freq = self.config.REFIT_FREQUENCY
        
        print(f"\nConfig:")
        print(f"  Min training: {min_train} days")
        print(f"  Refit: {refit_freq} days")
        print(f"  Assets: {', '.join(features_dict.keys())}")
        
        last_refit_idx = min_train
        fold_num = 0
        
        # WALK-FORWARD LOOP
        for i in range(min_train, len(common_dates)):
            current_date = common_dates.iloc[i]
            days_since_refit = i - last_refit_idx
            
            # === REFIT ===
            if (not self.hmms) or (days_since_refit >= refit_freq):
                fold_num += 1
                train_end = i
                
                print(f"\n{'='*80}")
                print(f"FOLD {fold_num}: Refitting at index {i}")
                print(f"{'='*80}")
                
                fold_info = {
                    'fold_num': fold_num,
                    'train_end': train_end,
                    'train_end_date': common_dates.iloc[train_end-1],
                    'asset_mappings': {}
                }
                
                # FIXED: Get QQQ forward returns for labeling
                target_train_dates = common_dates[:train_end].tolist()
                target_train_subset = target_data[target_data['Date'].isin(target_train_dates)].copy()
                target_train_subset['log_return'] = np.log(
                    target_train_subset['Close'] / target_train_subset['Close'].shift(1)
                )
                
                # Train HMM for each asset
                for asset, features_df in features_dict.items():
                    print(f"\n  Training {asset} HMM...")
                    
                    train_dates_list = common_dates[:train_end].tolist()
                    train_subset = features_df[features_df['Date'].isin(train_dates_list)].copy()
                    
                    # FIXED: Extract multi-feature vector [return, vol, momentum]
                    feature_cols = [f'{asset}_return_lag1', 
                                  f'{asset}_vol_lag1', 
                                  f'{asset}_momentum_lag1']
                    X_train = train_subset[feature_cols].dropna().values
                    
                    if len(X_train) < 100:
                        print(f"    WARNING: Insufficient data ({len(X_train)} points)")
                        continue
                    
                    # Standardize multi-feature vector
                    scaler = StandardScaler()
                    X_train_scaled = scaler.fit_transform(X_train)
                    
                    # Fit Student-t HMM
                    hmm_model = StudentTHMM(
                        n_states=self.config.HMM_N_STATES,
                        n_iter=self.config.HMM_N_ITER,
                        tol=self.config.HMM_CONVERGENCE_TOL,
                        random_state=self.config.HMM_RANDOM_STATE,
                        df_init=self.config.STUDENT_T_DF_INIT,
                        df_min=self.config.STUDENT_T_DF_MIN,
                        df_max=self.config.STUDENT_T_DF_MAX
                    )
                    
                    hmm_model.fit(X_train_scaled)
                    
                    print(f"    Converged: {hmm_model.converged_}, iter: {hmm_model.n_iter_}")
                    
                    # FIXED: Label states using QQQ forward returns
                    states_train = hmm_model.predict(X_train_scaled)
                    
                    # Align with target forward returns
                    train_dates_aligned = train_subset.dropna(subset=feature_cols)['Date'].tolist()
                    qqq_forward_returns = []
                    
                    for j, train_date in enumerate(train_dates_aligned):
                        if j + 1 < len(train_dates_aligned):
                            next_date = train_dates_aligned[j + 1]
                            # Get QQQ return at next date
                            qqq_next = target_train_subset[
                                target_train_subset['Date'] == next_date
                            ]['log_return']
                            
                            if not qqq_next.empty:
                                qqq_forward_returns.append(qqq_next.iloc[0])
                            else:
                                qqq_forward_returns.append(np.nan)
                        else:
                            qqq_forward_returns.append(np.nan)
                    
                    qqq_forward_returns = np.array(qqq_forward_returns)
                    
                    # Calculate mean QQQ forward return per state
                    state_returns = {}
                    for state in range(self.config.HMM_N_STATES):
                        mask = states_train == state
                        mask[-1] = False
                        
                        if mask.sum() > 0:
                            fwd_rets = qqq_forward_returns[mask]
                            fwd_rets = fwd_rets[~np.isnan(fwd_rets)]
                            if len(fwd_rets) > 0:
                                state_returns[state] = fwd_rets.mean()
                            else:
                                state_returns[state] = 0
                        else:
                            state_returns[state] = 0
                    
                    # Label: higher QQQ return → Risk-On
                    sorted_states = sorted(state_returns.items(), key=lambda x: x[1])
                    state_to_regime = {
                        sorted_states[0][0]: "Risk-Off",
                        sorted_states[1][0]: "Risk-On"
                    }
                    
                    # OVERFITTING FIX: Calculate predictive IR on HOLDOUT portion
                    # of training data, not in-sample. The HMM was trained on
                    # the full training set, so in-sample IR is overly optimistic.
                    # Use last 20% of training data as holdout for weight estimation.
                    
                    holdout_start = int(len(train_dates_aligned) * 0.80)
                    
                    if holdout_start > 20:
                        # Re-predict on holdout portion (HMM was trained on full set,
                        # but holdout data was near the end = most recent/relevant)
                        holdout_features = train_subset.dropna(subset=feature_cols).iloc[holdout_start:]
                        X_holdout = holdout_features[feature_cols].values
                        
                        if len(X_holdout) > 10:
                            X_holdout_scaled = scaler.transform(X_holdout)
                            states_holdout = hmm_model.predict(X_holdout_scaled)
                            
                            # Get regime signals for holdout
                            regime_signal_holdout = np.zeros(len(states_holdout))
                            for j, state in enumerate(states_holdout):
                                if state_to_regime[state] == "Risk-On":
                                    regime_signal_holdout[j] = 1.0
                                else:
                                    regime_signal_holdout[j] = -1.0
                            
                            # Get QQQ forward returns for holdout period
                            holdout_fwd_returns = qqq_forward_returns[holdout_start:]
                            
                            # Calculate signal returns on holdout
                            signal_returns = []
                            min_len = min(len(regime_signal_holdout) - 1, len(holdout_fwd_returns))
                            for j in range(min_len):
                                fwd_ret = holdout_fwd_returns[j]
                                if not np.isnan(fwd_ret):
                                    signal_returns.append(regime_signal_holdout[j] * fwd_ret)
                            
                            signal_returns = np.array(signal_returns)
                            
                            if len(signal_returns) > 10:
                                mean_return = signal_returns.mean()
                                std_return = signal_returns.std()
                                if std_return > 1e-6:
                                    information_ratio = mean_return / std_return * np.sqrt(252)
                                else:
                                    information_ratio = 0.0
                                weight = max(information_ratio, 0.1)
                            else:
                                weight = 0.5
                        else:
                            weight = 0.5
                    else:
                        weight = 0.5  # Default if insufficient data for holdout
                    
                    regime_mapping = RegimeMapping(
                        state_to_regime=state_to_regime,
                        mean_returns=state_returns,
                        training_end_date=common_dates.iloc[train_end-1],
                        asset=asset,
                        correlation_with_target=weight  # Now stores predictive IR
                    )
                    
                    self.hmms[asset] = (hmm_model, scaler)
                    self.regime_mappings[asset] = regime_mapping
                    
                    print(f"    Regime mapping (by QQQ fwd return):")
                    for state, regime in state_to_regime.items():
                        print(f"      State {state} → {regime}: "
                              f"QQQ fwd ret = {state_returns[state]:.6f}")
                    print(f"    Predictive IR: {weight:.3f} (used as consensus weight)")
                    
                    fold_info['asset_mappings'][asset] = {
                        'state_to_regime': state_to_regime.copy(),
                        'mean_returns': state_returns.copy(),
                        'predictive_ir': weight,
                        'weight': weight
                    }
                
                self.fold_history.append(fold_info)
                last_refit_idx = i
            
            # === PREDICT ===
            
            prob_risk_on_list = []
            weights_list = []
            
            for asset, features_df in features_dict.items():
                if asset not in self.hmms:
                    continue
                
                hmm_model, scaler = self.hmms[asset]
                regime_mapping = self.regime_mappings[asset]
                
                asset_current_data = features_df[features_df['Date'] == current_date]
                
                if asset_current_data.empty:
                    continue
                
                # FIXED: Extract multi-feature vector
                feature_cols = [f'{asset}_return_lag1', 
                              f'{asset}_vol_lag1', 
                              f'{asset}_momentum_lag1']
                X_current = asset_current_data[feature_cols].values
                
                if np.isnan(X_current).any() or len(X_current) == 0:
                    continue
                
                X_current_scaled = scaler.transform(X_current)
                
                probs = hmm_model.predict_proba(X_current_scaled)[0]
                
                prob_risk_on = 0.0
                for state in range(self.config.HMM_N_STATES):
                    if regime_mapping.state_to_regime[state] == "Risk-On":
                        prob_risk_on += probs[state]
                
                state_pred = hmm_model.predict(X_current_scaled)[0]
                regime_pred = regime_mapping.state_to_regime[state_pred]
                
                results.loc[results['Date'] == current_date, f'{asset}_regime'] = state_pred
                results.loc[results['Date'] == current_date, f'{asset}_regime_name'] = regime_pred
                results.loc[results['Date'] == current_date, f'{asset}_prob_risk_on'] = prob_risk_on
                results.loc[results['Date'] == current_date, f'{asset}_weight'] = regime_mapping.correlation_with_target
                
                prob_risk_on_list.append(prob_risk_on)
                weights_list.append(regime_mapping.correlation_with_target)
            
            # FIXED: Weighted consensus by correlation
            if len(prob_risk_on_list) > 0:
                weights_array = np.array(weights_list)
                probs_array = np.array(prob_risk_on_list)
                
                # Normalize weights
                weights_normalized = weights_array / weights_array.sum()
                
                # Weighted average
                consensus_prob = np.sum(probs_array * weights_normalized)
                
                # Graduated regime classification
                if consensus_prob >= 0.65:
                    final_regime = "Strong Risk-On"
                elif consensus_prob >= self.config.CONSENSUS_RISK_ON_THRESHOLD:
                    final_regime = "Moderate Risk-On"
                elif consensus_prob >= self.config.CONSENSUS_RISK_OFF_THRESHOLD:
                    final_regime = "Neutral"
                elif consensus_prob >= 0.35:
                    final_regime = "Moderate Risk-Off"
                else:
                    final_regime = "Strong Risk-Off"
                
                results.loc[results['Date'] == current_date, 'consensus_probability'] = consensus_prob
                results.loc[results['Date'] == current_date, 'final_regime'] = final_regime
        
        print(f"\n{'='*80}")
        print("DETECTION COMPLETE")
        print(f"{'='*80}")
        print(f"Folds: {len(self.fold_history)}")
        print(f"Predictions: {results['final_regime'].notna().sum()}")
        
        regime_counts = results['final_regime'].value_counts()
        print(f"\nRegime distribution:")
        for regime in regime_counts.index:
            count = regime_counts[regime]
            pct = count / len(results) * 100
            print(f"  {regime:20s}: {count:4d} days ({pct:5.1f}%)")
        
        print("="*80)
        
        return results


# ============================================================================
# DATA FETCHER
# ============================================================================

class DataFetcher:
    
    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        frame = download_ohlcv(ticker, start_date, end_date)
        if len(frame) < 100:
            raise ValueError(f"Only {len(frame)} rows for {ticker}; need at least 100.")
        return frame.rename_axis("Date").reset_index()[["Date", "Open", "High", "Low", "Close", "Volume"]]
    
    def fetch_all_data(self, config: StrategyConfig) -> Dict[str, pd.DataFrame]:
        print(f"\nFetching: {config.BACKTEST_START_DATE} to {config.BACKTEST_END_DATE}")
        
        buffer_start = (datetime.strptime(config.BACKTEST_START_DATE, '%Y-%m-%d') - 
                       timedelta(days=config.MIN_TRAINING_DAYS + 100)
                       ).strftime('%Y-%m-%d')
        
        all_tickers = (config.MACRO_ASSETS + 
                      [config.LONG_TICKER, config.HEDGE_TICKER, config.BENCHMARK_TICKER])
        all_tickers = list(set(all_tickers))
        
        data_dict = {}
        for ticker in all_tickers:
            try:
                df = self.fetch_historical_data(ticker, buffer_start, config.BACKTEST_END_DATE)
                data_dict[ticker] = df
            except:
                print(f"  ! Skipping {ticker}")
                continue
        
        if len(data_dict) < 4:
            raise ValueError(f"Insufficient data: {len(data_dict)} assets")
        
        print(f"\n✓ Fetched {len(data_dict)} assets")
        return data_dict


# ============================================================================
# BACKTEST ENGINE (Conceptually Fixed)
# ============================================================================

class BacktestEngine:
    """
    FIXED:
    1. Graduated exposure based on consensus strength
    2. Track actual entry prices for stop-loss
    3. Daily rebalancing (aligned with 1-day signal horizon)
    """
    
    def __init__(self, config: StrategyConfig):
        self.config = config
        self.trades = []
        self.stop_loss_triggers = []
        self.position_tracker = {}  # FIXED: Track entry prices
    
    def calculate_transaction_costs(self, trade_value: float) -> float:
        commission = abs(trade_value) * self.config.COMMISSION_RATE
        commission = max(commission, self.config.MIN_COMMISSION)
        spread = abs(trade_value) * self.config.BID_ASK_SPREAD
        impact = abs(trade_value) * self.config.MARKET_IMPACT
        return commission + spread + impact
    
    def calculate_stop_loss(self, ticker: str, date: pd.Timestamp,
                          data_dict: Dict) -> float:
        """
        Calculate CVaR-based stop loss with stability improvements.
        
        FIXED:
        - Minimum observations requirement
        - Winsorization to remove microstructure outliers
        - EWMA smoothing for stability
        """
        if ticker not in data_dict or date not in data_dict[ticker]['Date'].values:
            return -0.10
        
        df = data_dict[ticker]
        date_idx = df[df['Date'] == date].index[0]
        
        MIN_OBSERVATIONS = 40  # Minimum for reliable CVaR
        
        if date_idx < max(self.config.CVAR_LOOKBACK, MIN_OBSERVATIONS):
            return -0.10
        
        # Historical Open-to-Open returns
        hist_opens = df.loc[
            date_idx - self.config.CVAR_LOOKBACK:date_idx - 1, 'Open'
        ]
        hist_returns = hist_opens.pct_change().dropna()
        
        if len(hist_returns) < MIN_OBSERVATIONS:
            return -0.10
        
        # FIXED: Winsorize to remove microstructure outliers (1%/99%)
        lower_bound = hist_returns.quantile(0.01)
        upper_bound = hist_returns.quantile(0.99)
        hist_returns_winsorized = hist_returns.clip(lower=lower_bound, upper=upper_bound)
        
        # Calculate CVaR
        sorted_returns = np.sort(hist_returns_winsorized.values)
        cutoff_idx = int(len(sorted_returns) * (1 - self.config.CVAR_CONFIDENCE))
        
        if cutoff_idx >= 0:
            cvar_raw = sorted_returns[:cutoff_idx+1].mean()
        else:
            cvar_raw = sorted_returns[0]
        
        # FIXED: EWMA smoothing for stability (α=0.3)
        # Use previous CVaR if available
        if hasattr(self, 'prev_cvar'):
            alpha = 0.3
            cvar = alpha * cvar_raw + (1 - alpha) * self.prev_cvar
        else:
            cvar = cvar_raw
        
        self.prev_cvar = cvar
        
        # Apply buffer
        return cvar * self.config.STOP_LOSS_BUFFER
    
    def run_backtest(self, regime_predictions: pd.DataFrame,
                    data_dict: Dict[str, pd.DataFrame]) -> Dict:
        print("\n" + "="*80)
        print("BACKTEST (CONCEPTUALLY FIXED)")
        print("="*80)
        
        backtest_start = pd.to_datetime(self.config.BACKTEST_START_DATE)
        backtest_end = pd.to_datetime(self.config.BACKTEST_END_DATE)
        
        backtest_data = regime_predictions[
            (regime_predictions['Date'] >= backtest_start) &
            (regime_predictions['Date'] <= backtest_end)
        ].copy().reset_index(drop=True)
        
        print(f"Period: {backtest_data['Date'].min().date()} to {backtest_data['Date'].max().date()}")
        print(f"Days: {len(backtest_data)}")
        
        # Tracking
        portfolio_values = []
        daily_returns = []
        positions_held = []
        regime_history = []
        consensus_history = []
        leverage_history = []
        margin_costs_history = []
        benchmark_values = []
        benchmark_returns = []
        
        # State
        capital = self.config.INITIAL_CAPITAL
        prev_portfolio_value = capital
        prev_bench_value = capital
        qqq_shares = 0.0
        sh_shares = 0.0
        cash = capital
        
        # FIXED: Track entry prices
        qqq_entry_price = 0.0
        
        stop_loss_count = 0
        last_rebalance = -self.config.REBALANCE_FREQUENCY
        
        # Initialize benchmark
        if self.config.BENCHMARK_TICKER in data_dict:
            bench_df = data_dict[self.config.BENCHMARK_TICKER]
            first_bench_date = backtest_data['Date'].iloc[0]
            bench_match = bench_df[bench_df['Date'] == first_bench_date]
            if not bench_match.empty:
                bench_initial_price = bench_match.iloc[0]['Open']
                bench_shares = self.config.INITIAL_CAPITAL / bench_initial_price
            else:
                bench_shares = 0
        else:
            bench_shares = 0
        
        # BACKTEST LOOP
        for day_idx, row in backtest_data.iterrows():
            current_date = row['Date']
            
            if day_idx % 50 == 0:
                print(f"  Day {day_idx+1}/{len(backtest_data)}: {current_date.strftime('%Y-%m-%d')}")
            
            final_regime = row['final_regime']
            consensus_prob = row['consensus_probability']
            
            if pd.isna(final_regime):
                final_regime = "Neutral"
            if pd.isna(consensus_prob):
                consensus_prob = 0.5
            
            # Get Close prices
            qqq_close = None
            sh_close = None
            bench_close = None
            
            if self.config.LONG_TICKER in data_dict:
                qqq_data = data_dict[self.config.LONG_TICKER]
                qqq_match = qqq_data[qqq_data['Date'] == current_date]
                if not qqq_match.empty:
                    qqq_close = qqq_match.iloc[0]['Close']
            
            if self.config.HEDGE_TICKER in data_dict:
                sh_data = data_dict[self.config.HEDGE_TICKER]
                sh_match = sh_data[sh_data['Date'] == current_date]
                if not sh_match.empty:
                    sh_close = sh_match.iloc[0]['Close']
            
            if self.config.BENCHMARK_TICKER in data_dict:
                bench_df = data_dict[self.config.BENCHMARK_TICKER]
                bench_match = bench_df[bench_df['Date'] == current_date]
                if not bench_match.empty:
                    bench_close = bench_match.iloc[0]['Close']
            
            # Value holdings at Close prices
            qqq_value = qqq_shares * qqq_close if qqq_close else 0
            sh_value = sh_shares * sh_close if sh_close else 0
            
            # ACCOUNTING ORDER (documented for consistency):
            # 1. Apply margin financing cost on negative cash (if leveraged)
            # 2. Apply risk-free rate on positive cash
            # 3. Calculate portfolio value = cash + holdings
            # 4. This ensures costs are reflected before returns calculation
            
            margin_cost_today = 0.0
            if cash < 0:
                daily_margin_rate = self.config.MARGIN_RATE_ANNUAL / 252
                margin_cost_today = abs(cash) * daily_margin_rate
                cash -= margin_cost_today  # Reduces cash (more negative)
            
            # Risk-free rate on positive cash
            if day_idx > 0 and cash > 0:
                cash += cash * (self.config.RISK_FREE_RATE / 252)
            
            # Portfolio valuation AFTER accounting adjustments
            portfolio_value = cash + qqq_value + sh_value
            
            # Leverage
            gross_exposure = qqq_value + sh_value
            current_leverage = gross_exposure / portfolio_value if portfolio_value > 0 else 1.0
            
            # Returns
            if day_idx > 0:
                daily_return = (portfolio_value / prev_portfolio_value) - 1
            else:
                daily_return = 0
            
            # Benchmark
            if bench_close and bench_shares > 0:
                bench_value = bench_shares * bench_close
                if day_idx > 0:
                    bench_return = (bench_value / prev_bench_value) - 1
                else:
                    bench_return = 0
                prev_bench_value = bench_value
            else:
                bench_value = self.config.INITIAL_CAPITAL
                bench_return = 0
            
            # Store
            portfolio_values.append(portfolio_value)
            daily_returns.append(daily_return)
            positions_held.append(1 if qqq_shares > 0 else 0)
            regime_history.append(final_regime)
            consensus_history.append(consensus_prob)
            leverage_history.append(current_leverage)
            margin_costs_history.append(margin_cost_today)
            benchmark_values.append(bench_value)
            benchmark_returns.append(bench_return)
            
            prev_portfolio_value = portfolio_value
            
            # LOOK-AHEAD FIX: Stop-loss evaluated using Close[T] (known at signal time)
            # Previously used Open[T+1] which is unknown on day T.
            # Decision: evaluate stop at Close[T], execute liquidation at Open[T+1].
            if (self.config.STOP_LOSS_ENABLED and qqq_shares > 0 and 
                qqq_entry_price > 0 and qqq_close is not None and
                day_idx < len(backtest_data) - 1):
                
                # FIX: Use today's close (known) to evaluate stop condition
                projected_return = (qqq_close / qqq_entry_price) - 1
                
                stop_threshold = self.calculate_stop_loss(
                    self.config.LONG_TICKER, current_date, data_dict
                )
                
                if projected_return < stop_threshold:
                    # Execute at next day's open (correct execution timing)
                    next_date = backtest_data.iloc[day_idx + 1]['Date']
                    next_open = None
                    
                    if self.config.LONG_TICKER in data_dict:
                        qqq_data = data_dict[self.config.LONG_TICKER]
                        qqq_next = qqq_data[qqq_data['Date'] == next_date]
                        if not qqq_next.empty:
                            next_open = qqq_next.iloc[0]['Open']
                    
                    if next_open is not None:
                        liquidation_value = qqq_shares * next_open
                        cost = self.calculate_transaction_costs(liquidation_value)
                        pnl_realized = (next_open - qqq_entry_price) * qqq_shares
                        
                        self.stop_loss_triggers.append({
                            'date': current_date,
                            'execution_date': next_date,
                            'projected_return': projected_return,
                            'stop_threshold': stop_threshold,
                            'exit_price': next_open,
                            'entry_price': qqq_entry_price,
                            'pnl_realized': pnl_realized
                        })
                        
                        cash += liquidation_value - cost
                        qqq_shares = 0.0
                        qqq_entry_price = 0.0
                        
                        stop_loss_count += 1
                        
                        self.trades.append({
                            'date': current_date,
                            'ticker': self.config.LONG_TICKER,
                            'action': 'STOP_LOSS',
                            'shares': liquidation_value / next_open,
                            'price': next_open,
                            'cost': cost,
                            'regime': final_regime
                        })
                        
                        continue
            
            # REBALANCING (daily aligned with 1-day signal)
            days_since_rebal = day_idx - last_rebalance
            should_rebalance = (days_since_rebal >= self.config.REBALANCE_FREQUENCY)
            
            if should_rebalance and day_idx < len(backtest_data) - 1:
                next_date = backtest_data.iloc[day_idx + 1]['Date']
                
                # Get Open prices
                qqq_open = None
                sh_open = None
                
                if self.config.LONG_TICKER in data_dict:
                    qqq_data = data_dict[self.config.LONG_TICKER]
                    qqq_match = qqq_data[qqq_data['Date'] == next_date]
                    if not qqq_match.empty:
                        qqq_open = qqq_match.iloc[0]['Open']
                
                if self.config.HEDGE_TICKER in data_dict:
                    sh_data = data_dict[self.config.HEDGE_TICKER]
                    sh_match = sh_data[sh_data['Date'] == next_date]
                    if not sh_match.empty:
                        sh_open = sh_match.iloc[0]['Open']
                
                # FIXED: Graduated exposure based on consensus strength
                if consensus_prob >= 0.65:
                    target_qqq_alloc = self.config.STRONG_RISK_ON_EXPOSURE
                    target_sh_alloc = 0.0
                elif consensus_prob >= 0.55:
                    target_qqq_alloc = self.config.MODERATE_RISK_ON_EXPOSURE
                    target_sh_alloc = 0.0
                elif consensus_prob >= 0.45:
                    target_qqq_alloc = self.config.NEUTRAL_EXPOSURE
                    target_sh_alloc = 0.0
                elif consensus_prob >= 0.35:
                    target_qqq_alloc = self.config.DEFENSIVE_EXPOSURE
                    target_sh_alloc = 0.0
                else:
                    target_qqq_alloc = self.config.DEFENSIVE_EXPOSURE
                    target_sh_alloc = self.config.STRONG_RISK_OFF_HEDGE
                
                # Check leverage constraint
                total_target = target_qqq_alloc + target_sh_alloc
                if total_target > self.config.MAX_LEVERAGE:
                    scale = self.config.MAX_LEVERAGE / total_target
                    target_qqq_alloc *= scale
                    target_sh_alloc *= scale
                
                # Rebalance QQQ
                if qqq_open and qqq_open > 0:
                    target_qqq_value = portfolio_value * target_qqq_alloc
                    current_qqq_value = qqq_shares * qqq_open
                    trade_value = target_qqq_value - current_qqq_value
                    
                    if abs(trade_value) > portfolio_value * 0.01:
                        shares_to_trade = trade_value / qqq_open
                        cost = self.calculate_transaction_costs(trade_value)
                        
                        # FIXED: Update entry price tracking
                        if shares_to_trade > 0:  # Buy
                            # Weighted average entry price
                            total_value = qqq_shares * qqq_entry_price + shares_to_trade * qqq_open
                            total_shares = qqq_shares + shares_to_trade
                            qqq_entry_price = total_value / total_shares if total_shares > 0 else qqq_open
                        elif shares_to_trade < 0:  # Sell
                            # Keep entry price if partial exit, reset if full exit
                            if qqq_shares + shares_to_trade <= 0:
                                qqq_entry_price = 0.0
                        
                        qqq_shares += shares_to_trade
                        cash -= (trade_value + cost)
                        
                        self.trades.append({
                            'date': current_date,
                            'ticker': self.config.LONG_TICKER,
                            'action': 'BUY' if shares_to_trade > 0 else 'SELL',
                            'shares': abs(shares_to_trade),
                            'price': qqq_open,
                            'cost': cost,
                            'regime': final_regime
                        })
                
                # Rebalance SH
                if sh_open and sh_open > 0:
                    target_sh_value = portfolio_value * target_sh_alloc
                    current_sh_value = sh_shares * sh_open
                    trade_value = target_sh_value - current_sh_value
                    
                    if abs(trade_value) > portfolio_value * 0.01:
                        shares_to_trade = trade_value / sh_open
                        cost = self.calculate_transaction_costs(trade_value)
                        
                        sh_shares += shares_to_trade
                        cash -= (trade_value + cost)
                        
                        self.trades.append({
                            'date': current_date,
                            'ticker': self.config.HEDGE_TICKER,
                            'action': 'BUY' if shares_to_trade > 0 else 'SELL',
                            'shares': abs(shares_to_trade),
                            'price': sh_open,
                            'cost': cost,
                            'regime': final_regime
                        })
                
                last_rebalance = day_idx
        
        # METRICS
        print("\nCalculating metrics...")
        
        portfolio_df = pd.DataFrame({
            'Date': backtest_data['Date'],
            'Portfolio_Value': portfolio_values,
            'Daily_Returns': daily_returns,
            'Positions': positions_held,
            'Regime': regime_history,
            'Consensus_Prob': consensus_history,
            'Leverage': leverage_history,
            'Margin_Costs': margin_costs_history,
            'Benchmark_Value': benchmark_values,
            'Benchmark_Returns': benchmark_returns
        })
        
        final_value = portfolio_df['Portfolio_Value'].iloc[-1]
        total_return = (final_value / self.config.INITIAL_CAPITAL) - 1
        days = len(portfolio_df)
        annual_return = (1 + total_return) ** (252 / days) - 1 if days > 0 else 0
        
        returns = portfolio_df['Daily_Returns'].dropna()
        volatility = returns.std() * np.sqrt(252) if len(returns) > 0 else 0
        
        if volatility > 0.001:
            sharpe = (annual_return - self.config.RISK_FREE_RATE) / volatility
        else:
            sharpe = 0.0
        
        cumulative = (1 + returns).cumprod() if len(returns) > 0 else pd.Series([1])
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_dd = drawdown.min() if len(drawdown) > 0 else 0
        
        # Benchmark
        bench_returns_series = portfolio_df['Benchmark_Returns'].dropna()
        bench_final = portfolio_df['Benchmark_Value'].iloc[-1]
        bench_total_return = (bench_final / self.config.INITIAL_CAPITAL) - 1
        bench_annual_return = (1 + bench_total_return) ** (252 / days) - 1 if days > 0 else 0
        bench_volatility = bench_returns_series.std() * np.sqrt(252)
        
        if bench_volatility > 0.001:
            bench_sharpe = (bench_annual_return - self.config.RISK_FREE_RATE) / bench_volatility
        else:
            bench_sharpe = 0.0
        
        # Excess
        excess_returns = returns - bench_returns_series
        excess_annual = annual_return - bench_annual_return
        tracking_error = excess_returns.std() * np.sqrt(252) if len(excess_returns) > 0 else 0
        
        if tracking_error > 0.001:
            information_ratio = excess_annual / tracking_error
        else:
            information_ratio = 0.0
        
        total_margin_cost = portfolio_df['Margin_Costs'].sum()
        avg_leverage = portfolio_df['Leverage'].mean()
        
        metrics = {
            'final_value': final_value,
            'total_return': total_return,
            'annual_return': annual_return,
            'volatility': volatility,
            'sharpe_ratio': sharpe,
            'max_drawdown': max_dd,
            'num_trades': len(self.trades),
            'stop_losses': stop_loss_count,
            'total_margin_cost': total_margin_cost,
            'avg_leverage': avg_leverage,
            'benchmark_total_return': bench_total_return,
            'benchmark_annual_return': bench_annual_return,
            'benchmark_volatility': bench_volatility,
            'benchmark_sharpe': bench_sharpe,
            'excess_return': excess_annual,
            'tracking_error': tracking_error,
            'information_ratio': information_ratio
        }
        
        results = {
            'portfolio_history': portfolio_df,
            'trade_history': self.trades,
            'stop_loss_history': self.stop_loss_triggers,
            'metrics': metrics,
            'regime_predictions': regime_predictions,
            'data_dict': data_dict
        }
        
        print("\n" + "="*80)
        print("BACKTEST COMPLETE")
        print("="*80)
        print(f"Final: ${final_value:,.0f}")
        print(f"Total: {total_return:.1%} | Annual: {annual_return:.1%}")
        print(f"Sharpe: {sharpe:.2f} | Max DD: {max_dd:.1%}")
        print(f"Trades: {len(self.trades)} | Stops: {stop_loss_count}")
        print(f"Leverage: {avg_leverage:.2f}x | Margin: ${total_margin_cost:,.2f}")
        print(f"\nBenchmark ({self.config.BENCHMARK_TICKER}):")
        print(f"Annual: {bench_annual_return:.1%} | Sharpe: {bench_sharpe:.2f}")
        print(f"\nExcess: {excess_annual:.1%} | IR: {information_ratio:.2f}")
        print("="*80)
        
        return results


# ============================================================================
# VISUALIZATION
# ============================================================================

def create_dashboard(results: Dict, config: StrategyConfig):
    portfolio_df = results['portfolio_history']
    
    fig = plt.figure(figsize=(20, 14))
    gs = fig.add_gridspec(4, 2, hspace=0.3, wspace=0.3)
    
    # 1. Portfolio vs Benchmark
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(portfolio_df['Date'], portfolio_df['Portfolio_Value'],
            linewidth=2.5, color='#2E86AB', label='Strategy')
    ax1.plot(portfolio_df['Date'], portfolio_df['Benchmark_Value'],
            linewidth=2, color='#FF7F0E', alpha=0.8, label=f'{config.BENCHMARK_TICKER}')
    ax1.set_title('Multi-Asset Regime Sync (Conceptually Fixed)', 
                 fontsize=16, fontweight='bold')
    ax1.set_ylabel('Value ($)', fontsize=12)
    ax1.legend(fontsize=11)
    ax1.grid(alpha=0.3)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    
    # 2. Consensus
    ax2 = fig.add_subplot(gs[1, 0])
    colors = portfolio_df['Consensus_Prob'].apply(
        lambda x: '#2ca02c' if x > 0.65 else ('#d62728' if x < 0.35 else '#7f7f7f')
    )
    ax2.scatter(portfolio_df['Date'], portfolio_df['Consensus_Prob'],
               c=colors, alpha=0.6, s=20)
    # FIXED: Match config thresholds
    ax2.axhline(y=config.CONSENSUS_RISK_ON_THRESHOLD, color='green', 
               linestyle='--', alpha=0.5, label=f'Risk-On ({config.CONSENSUS_RISK_ON_THRESHOLD})')
    ax2.axhline(y=config.CONSENSUS_RISK_OFF_THRESHOLD, color='red', 
               linestyle='--', alpha=0.5, label=f'Risk-Off ({config.CONSENSUS_RISK_OFF_THRESHOLD})')
    ax2.set_title('Weighted Consensus', fontsize=14, fontweight='bold')
    ax2.set_ylabel('P(Risk-On)', fontsize=11)
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)
    
    # 3. Leverage
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.plot(portfolio_df['Date'], portfolio_df['Leverage'],
            linewidth=2, color='purple')
    ax3.axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
    ax3.axhline(y=config.MAX_LEVERAGE, color='red', linestyle='--', alpha=0.5)
    ax3.set_title('Leverage', fontsize=14, fontweight='bold')
    ax3.set_ylabel('Leverage (x)', fontsize=11)
    ax3.grid(alpha=0.3)
    
    # 4. Regime Distribution
    ax4 = fig.add_subplot(gs[2, 0])
    regime_counts = portfolio_df['Regime'].value_counts()
    ax4.bar(range(len(regime_counts)), regime_counts.values, color='steelblue')
    ax4.set_xticks(range(len(regime_counts)))
    ax4.set_xticklabels(regime_counts.index, rotation=45, ha='right')
    ax4.set_title('Regime Distribution', fontsize=14, fontweight='bold')
    ax4.set_ylabel('Days', fontsize=11)
    for i, v in enumerate(regime_counts.values):
        ax4.text(i, v, str(v), ha='center', va='bottom')
    ax4.grid(alpha=0.3, axis='y')
    
    # 5. Drawdown
    ax5 = fig.add_subplot(gs[2, 1])
    returns = portfolio_df['Daily_Returns']
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = ((cumulative - running_max) / running_max) * 100
    ax5.fill_between(portfolio_df['Date'], drawdown, 0, alpha=0.5, color='red')
    ax5.plot(portfolio_df['Date'], drawdown, color='darkred', linewidth=1.5)
    ax5.set_title('Drawdown', fontsize=14, fontweight='bold')
    ax5.set_ylabel('Drawdown (%)', fontsize=11)
    ax5.grid(alpha=0.3)
    
    # 6. Summary
    ax6 = fig.add_subplot(gs[3, :])
    ax6.axis('off')
    
    metrics = results['metrics']
    
    summary = f"""
    CONCEPTUALLY CORRECTED MULTI-ASSET REGIME SYNCHRONIZATION
    {'='*100}

    STRATEGY RESULTS:
    • Total: {metrics['total_return']:.2%} | Annual: {metrics['annual_return']:.2%} | Volatility: {metrics['volatility']:.2%}
    • Sharpe: {metrics['sharpe_ratio']:.2f} | Max DD: {metrics['max_drawdown']:.2%}
    • Trades: {metrics['num_trades']} | Stop Losses: {metrics['stop_losses']}
    • Avg Leverage: {metrics['avg_leverage']:.2f}x | Margin Cost: ${metrics['total_margin_cost']:,.2f}

    BENCHMARK ({config.BENCHMARK_TICKER}):
    • Annual: {metrics['benchmark_annual_return']:.2%} | Sharpe: {metrics['benchmark_sharpe']:.2f}

    RELATIVE:
    • Excess Return: {metrics['excess_return']:.2%} | Tracking Error: {metrics['tracking_error']:.2%}
    • Information Ratio: {metrics['information_ratio']:.2f}

    CONCEPTUAL FIXES APPLIED:
    ✓ Regime labels use QQQ forward returns (not each asset's own)
    ✓ Multi-feature HMM: [return, volatility, momentum]
    ✓ Weighted consensus by correlation to QQQ
    ✓ Signal horizon = rebalance frequency (1 day)
    ✓ Benchmark = QQQ (matches trading instrument)
    ✓ Entry price tracking for accurate stop-loss
    ✓ Graduated exposure (5 levels vs binary on/off)
    """
    
    ax6.text(0.05, 0.95, summary, 
            transform=ax6.transAxes, fontsize=9,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=1', facecolor='lightblue', alpha=0.8))
    
    plt.suptitle(f'Multi-Asset Regime Sync - Fixed (2013-2018)', 
                fontsize=18, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()


# ============================================================================
# MAIN
# ============================================================================

def main():
    print("="*80)
    print("MULTI-ASSET REGIME SYNCHRONIZATION (CONCEPTUALLY CORRECTED)")
    print("="*80)
    
    config = StrategyConfig()
    
    # 1. Fetch data
    print("\n" + "="*80)
    print("STEP 1: DATA")
    print("="*80)
    
    fetcher = DataFetcher()
    data_dict = fetcher.fetch_all_data(config)
    
    # 2. Features
    print("\n" + "="*80)
    print("STEP 2: FEATURES")
    print("="*80)
    
    features_dict = {}
    for asset in config.MACRO_ASSETS:
        if asset in data_dict:
            print(f"\nCalculating features for {asset}...")
            features_df = FeatureEngine.prepare_features(
                data_dict[asset], asset, config
            )
            features_dict[asset] = features_df
    
    if len(features_dict) < 2:
        raise ValueError("Need at least 2 macro assets")
    
    # 3. Regime detection (with QQQ as target)
    print("\n" + "="*80)
    print("STEP 3: REGIME DETECTION")
    print("="*80)
    
    # FIXED: Pass QQQ data for forward return calculation
    target_data = data_dict[config.LONG_TICKER]
    
    detector = WalkForwardMultiAssetDetector(config)
    regime_predictions = detector.walk_forward_predict(features_dict, target_data)
    
    # 4. Backtest
    print("\n" + "="*80)
    print("STEP 4: BACKTEST")
    print("="*80)
    
    engine = BacktestEngine(config)
    results = engine.run_backtest(regime_predictions, data_dict)
    
    # 5. Visualization
    print("\n" + "="*80)
    print("STEP 5: VISUALIZATION")
    print("="*80)
    
    create_dashboard(results, config)
    
    print("\n" + "="*80)
    print("COMPLETE")
    print("="*80)
    
    return results


if __name__ == "__main__":
    results = main()
